# Phase 10.5 — Ranking Pipeline Integration

Simulated production path: RRF pool → hard filter → features → normalize → baseline rank.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.baseline_config import BaselineRankingConfig
from productiq.ranking.product_context import RankingProductContext
from productiq.ranking.product_context_provider import InMemoryRankingProductContextProvider
from productiq.representation.builder import build_product_representation
from productiq.representation.filtering import build_filtering_representation_from_canonical
from productiq.representation.query_contract import build_query_representation
from productiq.retrieval.catalog_candidate_filter import InMemoryCatalogCandidateFilter
from productiq.retrieval.contracts import RetrievalRequest
from productiq.retrieval.production_config import ProductionRetrievalConfig
from productiq.retrieval.production_pipeline import ProductionRetrievalPipeline
from productiq.retrieval.production_ranking_config import ProductionRankingStageConfig
from productiq.retrieval.ranking_integration import run_baseline_ranking_stage
from productiq.retrieval.rrf_fusion import fused_candidate_from_rrf
from tests.database.catalog_fixtures import make_product_record

In [ ]:
def fused(product_id: str, fusion: float) -> object:
    return fused_candidate_from_rrf(
        product_id,
        fusion_score=fusion,
        bm25_rank=1,
        vector_rank=1,
        bm25_score=1.0,
        vector_score=0.5,
    )


class StaticRRF:
    def __init__(self, ranking: tuple[str, ...]) -> None:
        self.ranking = ranking

    def retrieve(self, request):
        from productiq.retrieval.contracts import RetrievalResponse

        scores = [0.9 - index * 0.05 for index in range(len(self.ranking))]
        candidates = tuple(
            fused(pid, score) for pid, score in zip(self.ranking, scores, strict=True)
        )
        return RetrievalResponse(candidates=candidates[: request.top_k], metadata=None)


records = {
    "P1": make_product_record(product_id="P1", brand_normalized="nike"),
    "P2": make_product_record(product_id="P2", brand_normalized="adidas"),
    "P3": make_product_record(product_id="P3", brand_normalized="nike"),
}
filtering_map = {
    pid: build_filtering_representation_from_canonical(record) for pid, record in records.items()
}
contexts = {
    pid: RankingProductContext(
        product_id=pid,
        product=build_product_representation(record),
        filtering=filtering_map[pid],
    )
    for pid, record in records.items()
}
pipeline = ProductionRetrievalPipeline(
    rrf_retriever=StaticRRF(("P1", "P2", "P3")),
    catalog_filter=InMemoryCatalogCandidateFilter(filtering_map),
    config=ProductionRetrievalConfig(candidate_pool_top_k=10),
    product_context_provider=InMemoryRankingProductContextProvider(contexts),
    ranking_stage=ProductionRankingStageConfig(baseline_config=BaselineRankingConfig()),
)
query = build_query_representation("nike shoes")
request = RetrievalRequest(query=query, top_k=2)
ranked = pipeline.retrieve_ranked(request)
ranked.model_dump(mode="json")

In [ ]:
# Direct stage call on a filtered candidate tuple (no live DB / indexes)
filtered = (fused("P1", 0.8), fused("P3", 0.6))
stage = run_baseline_ranking_stage(
    query=query,
    filtered_candidates=filtered,
    top_k=2,
    product_context_provider=InMemoryRankingProductContextProvider(contexts),
    baseline_config=BaselineRankingConfig(),
)
{
    "ranked_ids": [row.product_id for row in stage.response.ranked_candidates],
    "timings_ms": stage.timings_ms.model_dump(),
}